# Automated DCF Valuation Engine — Built from Live SEC EDGAR Filings

**Author:** Jaik Wolfe  
**Type:** Intermediate-level quantitative finance project (Google Colab)

This notebook pulls a public company's real financial statements directly from the **SEC EDGAR XBRL API** (no manual copy-pasting from 10-Ks), builds a simplified 3-statement forecast, and produces a full **Discounted Cash Flow (DCF) valuation** with a WACC/terminal-growth sensitivity table — the same core workflow used in equity research, investment banking, and wealth management to estimate a stock's intrinsic value.

**How to use it:** set `TICKER` and `SEC_USER_AGENT` in the Configuration cell below, then Runtime → Run all.

## Project Overview

Given a stock ticker, this notebook automatically:
1. Looks up the company's CIK and pulls its full XBRL financial history from SEC EDGAR.
2. Cleans and assembles a 6-year historical financial statement (revenue, margins, D&A, capex, tax rate).
3. Pulls live market data (price, market cap, beta, debt, cash) from Yahoo Finance and the risk-free rate from FRED.
4. Computes a CAPM-based cost of equity and a full WACC.
5. Projects unlevered free cash flow forward with a fading growth rate.
6. Discounts the projected FCFs and a Gordon-growth terminal value back to the present.
7. Outputs an implied share price, a WACC × terminal-growth sensitivity table, and a comparison to the current market price.

## Real-World Finance Use Case

The DCF is the single most widely used intrinsic-valuation framework in equity research, investment banking, and fundamental/wealth-management investing. Analysts build this exact model — historical financials, forward FCF projection, WACC, terminal value, sensitivity table — every time they need to answer *"what is this stock actually worth, independent of what the market is paying for it?"* Automating the data-ingestion step (rather than retyping numbers out of a PDF 10-K) mirrors what financial data vendors and internal bank tooling do to speed up analyst workflows, and is exactly the kind of applied, real-data project that stands out over a textbook Excel DCF.

## System Architecture

```
┌──────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│   SEC EDGAR API   │     │  Yahoo Finance    │     │    FRED (CSV)     │
│ (XBRL financials) │     │ (price/cap/beta)  │     │ (risk-free rate)  │
└─────────┬─────────┘     └─────────┬─────────┘     └─────────┬─────────┘
          │                         │                          │
          ▼                         ▼                          ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │                    Data Layer (requests / yfinance)              │
   │   get_cik() → get_company_facts() → build_historicals()         │
   │   get_market_data()               → get_risk_free_rate()        │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │                    Modeling Layer (pandas / numpy)               │
   │   compute_wacc() → project_fcf() → dcf_valuation()              │
   │                                  → sensitivity_table()           │
   └───────────────────────────────┬───────────────────────────────-┘
                                    ▼
   ┌─────────────────────────────────────────────────────────────────┐
   │              Presentation Layer (matplotlib / plotly)            │
   │   historical chart · FCF chart · sensitivity heatmap · summary   │
   └─────────────────────────────────────────────────────────────────┘
```

## Required APIs and Data Sources

| Source | What it provides | Auth needed? |
|---|---|---|
| **SEC EDGAR** `data.sec.gov/api/xbrl/companyfacts` | Historical revenue, income, D&A, capex, tax, interest expense (from real 10-K filings) | No key, but **requires a descriptive `User-Agent`** (name + email) — SEC blocks/rate-limits anonymous requests |
| **Yahoo Finance** (`yfinance`) | Current price, market cap, shares outstanding, beta, total debt, cash | No key (unofficial wrapper — no SLA, values are best-effort) |
| **FRED** (public CSV endpoint) | 10-Year Treasury yield as the risk-free rate | No key needed via the public `fredgraph.csv` endpoint |

## Required Python Libraries

`requests`, `pandas`, `numpy`, `yfinance`, `matplotlib`, `plotly` — all pip-installable, installed in the first code cell.

## Folder/File Structure

Even though this runs as a single Colab notebook, it's organized like a small package so every piece is reusable:

```
DCF_Valuation_SEC_EDGAR.ipynb
├── Cell: Setup & installs
├── Cell: Configuration (TICKER, USER_AGENT, assumptions)
├── Section 1 — SEC EDGAR data pipeline      (get_cik, get_company_facts)
├── Section 2 — Historical financials        (annual_series, build_historicals)
├── Section 3 — Market data & risk-free rate (get_market_data, get_risk_free_rate)
├── Section 4 — Cost of capital / WACC       (compute_wacc)
├── Section 5 — FCF projection               (project_fcf)
├── Section 6 — DCF valuation                (dcf_valuation)
├── Section 7 — Sensitivity analysis         (sensitivity_table)
├── Section 8 — Visualizations               (4 charts)
└── Section 9 — Final summary report
```
If you ever move this out of Colab, each `## Section` maps cleanly onto its own module: `edgar_client.py`, `financials.py`, `market_data.py`, `valuation.py`, `viz.py`.

## Setup

Install dependencies (safe to re-run; Colab already has most of these).

In [ ]:
%pip install -q requests pandas numpy yfinance matplotlib plotly

## Configuration

**Edit these two values before running:**
- `TICKER` — the stock you want to value.
- `SEC_USER_AGENT` — SEC EDGAR requires a real, descriptive User-Agent (`"Your Name your.email@example.com"`). Anonymous or generic requests get blocked or rate-limited — this is a real, well-known SEC EDGAR gotcha, not a formality.

In [ ]:
import time
import warnings
from dataclasses import dataclass

import requests
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ----- USER-EDITABLE CONFIG -----------------------------------------------
TICKER = "AAPL"                                   # any US-listed filer with 10-Ks on EDGAR
SEC_USER_AGENT = "Jaik Wolfe jaikwolfe@outlook.com"  # REQUIRED: replace with your own name + email
PROJECTION_YEARS = 5                               # explicit forecast horizon
TERMINAL_GROWTH = 0.025                            # long-run (Gordon growth) terminal growth rate
EQUITY_RISK_PREMIUM = 0.045                        # market equity risk premium assumption for CAPM
DEFAULT_CREDIT_SPREAD = 0.015                      # fallback spread over the risk-free rate for cost of debt
HISTORY_YEARS = 6                                  # years of historical financials to pull
# ----------------------------------------------------------------------------

if "@example.com" in SEC_USER_AGENT or not SEC_USER_AGENT.strip():
    raise ValueError(
        "Set SEC_USER_AGENT to your own name and email before running — "
        "SEC EDGAR blocks generic/placeholder User-Agent strings (see sec.gov/os/webmaster-faq#developers)."
    )

HEADERS = {"User-Agent": SEC_USER_AGENT}
TICKER = TICKER.strip().upper()
print(f"Configured for {TICKER}. Requests will identify as: '{SEC_USER_AGENT}'")

## Section 1 — SEC EDGAR Data Pipeline

SEC EDGAR's `company_tickers.json` maps a ticker to its 10-digit **CIK** (Central Index Key), which is then used to fetch that company's entire XBRL financial history from `data.sec.gov/api/xbrl/companyfacts`. Both calls are wrapped with clear error messages and a small retry loop for transient rate-limiting (HTTP 429).

In [ ]:
def get_cik(ticker: str) -> str:
    """Resolve a stock ticker to a zero-padded 10-digit SEC CIK."""
    resp = requests.get(
        "https://www.sec.gov/files/company_tickers.json", headers=HEADERS, timeout=20
    )
    resp.raise_for_status()
    for row in resp.json().values():
        if row["ticker"].upper() == ticker.upper():
            return str(row["cik_str"]).zfill(10)
    raise ValueError(
        f"Ticker '{ticker}' was not found in SEC's company_tickers.json. "
        "Double-check the ticker, or note that not every listed company files 10-Ks with the SEC "
        "(e.g. foreign private issuers filing 20-F instead)."
    )


def get_company_facts(cik: str, max_retries: int = 3) -> dict:
    """Fetch a company's full XBRL 'company facts' JSON, retrying on rate limits."""
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    last_status = None
    for attempt in range(max_retries):
        resp = requests.get(url, headers=HEADERS, timeout=30)
        last_status = resp.status_code
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 429:  # rate-limited — back off and retry
            time.sleep(1.5 * (attempt + 1))
            continue
        if resp.status_code == 403:
            raise RuntimeError(
                "SEC EDGAR returned 403 Forbidden — this almost always means the User-Agent header "
                "was rejected. Make sure SEC_USER_AGENT includes a real name and email."
            )
        resp.raise_for_status()
    raise RuntimeError(f"SEC EDGAR request failed after {max_retries} retries (last status: {last_status}).")


cik = get_cik(TICKER)
print(f"{TICKER} CIK: {cik}")
company_facts = get_company_facts(cik)
print(f"Pulled XBRL facts for: {company_facts.get('entityName', TICKER)}")

## Section 2 — Parsing XBRL Into Clean Annual Financials

XBRL tags are messy in practice: companies restate prior years (10-K/A), switch tag names when accounting standards change (e.g. `Revenues` → `RevenueFromContractWithCustomerExcludingAssessedTax` after ASC 606), and mix annual with quarterly contexts under the same tag. `annual_series()` filters to full-year (~350-380 day) periods from 10-K filings only and keeps the most recently filed value per fiscal year-end to handle restatements. `first_available_series()` tries a list of tag names in priority order so the pipeline works across companies and accounting-standard eras without hardcoding one tag.

In [ ]:
REVENUE_TAGS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",  # modern ASC 606 tag (most filers, 2018+)
    "Revenues",                                              # legacy tag (pre-2018 filings)
    "SalesRevenueNet",                                       # older alternative tag
]
DA_TAGS = [
    "DepreciationDepletionAndAmortization",
    "DepreciationAmortizationAndAccretionNet",
    "DepreciationAndAmortization",
]
CAPEX_TAGS = [
    "PaymentsToAcquirePropertyPlantAndEquipment",
    "PaymentsToAcquireProductiveAssets",
]


def annual_series(usgaap: dict, tag: str) -> pd.Series:
    """Extract one XBRL us-gaap tag as a clean annual (10-K, full fiscal year) time series."""
    if tag not in usgaap:
        return pd.Series(dtype=float)
    entries = usgaap[tag]["units"].get("USD", [])
    rows = []
    for e in entries:
        if e.get("form") != "10-K":
            continue
        start, end = e.get("start"), e.get("end")
        if not start or not end:
            continue  # instant (balance-sheet) facts have no 'start' — skip for flow items
        days = (pd.Timestamp(end) - pd.Timestamp(start)).days
        if not (300 <= days <= 400):
            continue  # keep full fiscal years only, drop quarterly/partial-period contexts
        rows.append({"end": end, "val": e["val"], "filed": e.get("filed", "")})
    if not rows:
        return pd.Series(dtype=float)
    # Keep the most recently FILED value for each fiscal year-end to handle 10-K/A restatements
    df = (
        pd.DataFrame(rows)
        .sort_values("filed")
        .drop_duplicates("end", keep="last")
        .sort_values("end")
    )
    series = df.set_index("end")["val"]
    series.index = pd.to_datetime(series.index)
    return series


def first_available_series(usgaap: dict, tags: list) -> pd.Series:
    """Merge a priority list of tag candidates into one series (first tag wins on overlap)."""
    combined = pd.Series(dtype=float)
    for tag in tags:
        combined = annual_series(usgaap, tag).combine_first(combined)
    return combined.sort_index()


def build_historicals(facts: dict, years: int = HISTORY_YEARS) -> pd.DataFrame:
    """Assemble a clean historical financials + ratio table from raw XBRL company facts."""
    if "us-gaap" not in facts.get("facts", {}):
        raise RuntimeError("This filer has no us-gaap XBRL facts (may file under IFRS or a different taxonomy).")
    usgaap = facts["facts"]["us-gaap"]

    df = pd.DataFrame({
        "revenue": first_available_series(usgaap, REVENUE_TAGS),
        "operating_income": annual_series(usgaap, "OperatingIncomeLoss"),
        "net_income": annual_series(usgaap, "NetIncomeLoss"),
        "d_and_a": first_available_series(usgaap, DA_TAGS),
        "capex": first_available_series(usgaap, CAPEX_TAGS),
        "pretax_income": annual_series(
            usgaap, "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"
        ),
        "tax_expense": annual_series(usgaap, "IncomeTaxExpenseBenefit"),
        "interest_expense": annual_series(usgaap, "InterestExpense"),
    }).sort_index()
    # Combining Series from combine_first()/dict-construction can silently downgrade a
    # DatetimeIndex to a plain object Index on some pandas versions — force it back so
    # downstream code (e.g. historicals.index.year in the charts) can always rely on it.
    df.index = pd.to_datetime(df.index)

    if df["revenue"].dropna().empty:
        raise RuntimeError(
            f"No annual revenue tag found for {TICKER} under any of {REVENUE_TAGS}. "
            "This filer may use a non-standard revenue tag — inspect company_facts['facts']['us-gaap'].keys()."
        )

    df = df.tail(years).copy()
    df["revenue_growth"] = df["revenue"].pct_change()
    df["operating_margin"] = (df["operating_income"] / df["revenue"]).replace([np.inf, -np.inf], np.nan)
    df["da_pct_revenue"] = (df["d_and_a"] / df["revenue"]).replace([np.inf, -np.inf], np.nan)
    df["capex_pct_revenue"] = (df["capex"] / df["revenue"]).replace([np.inf, -np.inf], np.nan)
    df["effective_tax_rate"] = (df["tax_expense"] / df["pretax_income"]).clip(0, 0.5)

    if len(df.dropna(subset=["revenue"])) < 2:
        raise RuntimeError(f"Only {len(df)} year(s) of usable history found for {TICKER} — too little to project from.")

    return df


historicals = build_historicals(company_facts)
display(historicals[["revenue", "operating_income", "net_income", "revenue_growth", "operating_margin"]])

## Section 3 — Market Data & Risk-Free Rate

Live price, market cap, beta, debt, and cash come from Yahoo Finance via `yfinance` (an unofficial, community-maintained wrapper — treat fields as best-effort, not a guaranteed SLA-backed feed). The risk-free rate uses FRED's public `fredgraph.csv` endpoint for the 10-Year Treasury yield (`DGS10`) — no API key required, unlike the `fredapi` package.

In [ ]:
def get_market_data(ticker: str) -> dict:
    """Pull live price, market cap, beta, debt, and cash for a ticker from Yahoo Finance."""
    info = yf.Ticker(ticker).info
    required = ["marketCap", "sharesOutstanding", "currentPrice"]
    missing = [k for k in required if not info.get(k)]
    if missing:
        raise RuntimeError(
            f"yfinance did not return required field(s) {missing} for {ticker}. "
            "This can happen for illiquid tickers, ADRs, or a temporary data-provider hiccup — try again shortly."
        )
    return {
        "market_cap": info["marketCap"],
        "shares_outstanding": info["sharesOutstanding"],
        "current_price": info["currentPrice"],
        "beta": info.get("beta") or 1.0,          # fall back to a market-average beta if missing
        "total_debt": info.get("totalDebt") or 0,
        "total_cash": info.get("totalCash") or 0,
    }


def get_risk_free_rate() -> float:
    """Latest 10-Year Treasury yield (DGS10) from FRED's public CSV endpoint, as a decimal."""
    resp = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10", timeout=20)
    resp.raise_for_status()
    rows = [r for r in resp.text.strip().split("\n")[1:] if r and r.split(",")[-1] != "."]
    if not rows:
        raise RuntimeError("FRED returned no usable DGS10 observations.")
    return float(rows[-1].split(",")[-1]) / 100


market = get_market_data(TICKER)
risk_free_rate = get_risk_free_rate()

print(f"Current price:        ${market['current_price']:,.2f}")
print(f"Market cap:           ${market['market_cap']:,.0f}")
print(f"Beta:                 {market['beta']:.2f}")
print(f"Total debt:           ${market['total_debt']:,.0f}")
print(f"Total cash:           ${market['total_cash']:,.0f}")
print(f"10Y risk-free rate:   {risk_free_rate:.2%}")

## Section 4 — Cost of Capital: CAPM Cost of Equity & WACC

Cost of equity uses CAPM: `Rf + beta * equity_risk_premium`. Pre-tax cost of debt is estimated from the most recent reported interest expense over total debt when available, floored at the risk-free rate and capped at `risk-free + 8%` as a sanity bound (falls back to `risk-free + DEFAULT_CREDIT_SPREAD` if the filer doesn't break out interest expense). WACC blends the two by market-value weights.

In [ ]:
def compute_wacc(hist: pd.DataFrame, mkt: dict, rf: float) -> dict:
    """CAPM cost of equity, an interest-expense-implied cost of debt, and market-value-weighted WACC."""
    cost_of_equity = rf + mkt["beta"] * EQUITY_RISK_PREMIUM

    interest_hist = hist["interest_expense"].dropna()
    debt_for_rate = mkt["total_debt"] or 1  # avoid a zero-division for debt-free companies
    if not interest_hist.empty and mkt["total_debt"]:
        pretax_cost_of_debt = float(interest_hist.iloc[-1]) / debt_for_rate
    else:
        pretax_cost_of_debt = rf + DEFAULT_CREDIT_SPREAD
    pretax_cost_of_debt = min(max(pretax_cost_of_debt, rf), rf + 0.08)  # sanity clamp

    tax_rate = hist["effective_tax_rate"].dropna().tail(3).mean()
    tax_rate = 0.21 if pd.isna(tax_rate) else float(tax_rate)  # 0.21 = US statutory corporate rate fallback
    after_tax_cost_of_debt = pretax_cost_of_debt * (1 - tax_rate)

    E, D = mkt["market_cap"], mkt["total_debt"]
    V = E + D
    wacc = (E / V) * cost_of_equity + (D / V) * after_tax_cost_of_debt if V else cost_of_equity

    return {
        "risk_free_rate": rf,
        "cost_of_equity": cost_of_equity,
        "pretax_cost_of_debt": pretax_cost_of_debt,
        "after_tax_cost_of_debt": after_tax_cost_of_debt,
        "tax_rate": tax_rate,
        "wacc": wacc,
    }


wacc_inputs = compute_wacc(historicals, market, risk_free_rate)
print(f"Cost of equity (CAPM):     {wacc_inputs['cost_of_equity']:.2%}")
print(f"After-tax cost of debt:    {wacc_inputs['after_tax_cost_of_debt']:.2%}")
print(f"Effective tax rate used:   {wacc_inputs['tax_rate']:.2%}")
print(f"--> WACC:                  {wacc_inputs['wacc']:.2%}")

## Section 5 — Free Cash Flow Projection

Revenue growth fades linearly from the trailing 3-year average growth rate toward `TERMINAL_GROWTH` over `PROJECTION_YEARS`. Operating margin, D&A %, and capex % of revenue are held at their trailing 3-year averages. Unlevered FCF = `EBIT * (1 - tax) + D&A - Capex - ΔNWC`, where the change in net working capital is simplified as 10% of the year-over-year revenue change — a common simplifying assumption, called out explicitly here rather than hidden.

In [ ]:
def project_fcf(hist: pd.DataFrame, wacc_inputs: dict, years: int = PROJECTION_YEARS) -> pd.DataFrame:
    """Project unlevered free cash flow forward using fading growth and historical-average margins."""
    recent_growth = hist["revenue_growth"].dropna().tail(3)
    start_growth = float(recent_growth.mean()) if not recent_growth.empty else 0.05
    start_growth = float(np.clip(start_growth, -0.05, 0.30))  # guard against outlier/one-off growth spikes

    op_margin = float(hist["operating_margin"].dropna().tail(3).mean())
    da_pct = float(hist["da_pct_revenue"].dropna().tail(3).mean())
    capex_pct = float(hist["capex_pct_revenue"].dropna().tail(3).mean())
    tax_rate = wacc_inputs["tax_rate"]
    last_revenue = float(hist["revenue"].dropna().iloc[-1])
    nwc_pct_of_delta_rev = 0.10  # simplifying assumption — documented, not hidden

    rows, revenue = [], last_revenue
    for yr in range(1, years + 1):
        g = start_growth + (TERMINAL_GROWTH - start_growth) * (yr / years)  # linear fade to terminal growth
        prev_revenue, revenue = revenue, revenue * (1 + g)
        ebit = revenue * op_margin
        nopat = ebit * (1 - tax_rate)
        da = revenue * da_pct
        capex = revenue * capex_pct
        delta_nwc = (revenue - prev_revenue) * nwc_pct_of_delta_rev
        fcf = nopat + da - capex - delta_nwc
        rows.append({
            "year": yr, "revenue": revenue, "growth": g, "ebit": ebit, "nopat": nopat,
            "d_and_a": da, "capex": capex, "delta_nwc": delta_nwc, "fcf": fcf,
        })
    return pd.DataFrame(rows)


fcf_projection = project_fcf(historicals, wacc_inputs)
display(fcf_projection.style.format({
    "revenue": "${:,.0f}", "growth": "{:.2%}", "ebit": "${:,.0f}", "nopat": "${:,.0f}",
    "d_and_a": "${:,.0f}", "capex": "${:,.0f}", "delta_nwc": "${:,.0f}", "fcf": "${:,.0f}",
}))

## Section 6 — Discounted Cash Flow Valuation

Discounts each projected year's FCF and a Gordon-growth terminal value back to the present at WACC, then bridges Enterprise Value → Equity Value → implied price per share.

In [ ]:
def dcf_valuation(fcf_df: pd.DataFrame, wacc: float, terminal_growth: float, mkt: dict) -> dict:
    """Discount projected FCFs + Gordon-growth terminal value to an implied share price."""
    if wacc <= terminal_growth:
        raise ValueError(
            f"WACC ({wacc:.2%}) must exceed the terminal growth rate ({terminal_growth:.2%}) "
            "for a Gordon-growth terminal value — lower TERMINAL_GROWTH or check the WACC inputs."
        )
    discount_factors = np.array([1 / (1 + wacc) ** row.year for row in fcf_df.itertuples()])
    pv_fcf = float((fcf_df["fcf"].values * discount_factors).sum())

    terminal_fcf = fcf_df["fcf"].iloc[-1] * (1 + terminal_growth)
    terminal_value = terminal_fcf / (wacc - terminal_growth)
    pv_terminal_value = float(terminal_value * discount_factors[-1])

    enterprise_value = pv_fcf + pv_terminal_value
    equity_value = enterprise_value - mkt["total_debt"] + mkt["total_cash"]
    implied_price = equity_value / mkt["shares_outstanding"]

    return {
        "pv_fcf": pv_fcf, "terminal_value": terminal_value, "pv_terminal_value": pv_terminal_value,
        "enterprise_value": enterprise_value, "equity_value": equity_value, "implied_price": implied_price,
    }


valuation = dcf_valuation(fcf_projection, wacc_inputs["wacc"], TERMINAL_GROWTH, market)
upside = valuation["implied_price"] / market["current_price"] - 1

print(f"PV of explicit FCFs:        ${valuation['pv_fcf']:,.0f}")
print(f"PV of terminal value:       ${valuation['pv_terminal_value']:,.0f}")
print(f"Enterprise value:           ${valuation['enterprise_value']:,.0f}")
print(f"Equity value:               ${valuation['equity_value']:,.0f}")
print(f"Implied share price:        ${valuation['implied_price']:,.2f}")
print(f"Current market price:       ${market['current_price']:,.2f}")
print(f"Implied upside/(downside):  {upside:+.1%}")

## Section 7 — Sensitivity Analysis: WACC × Terminal Growth

No real DCF is delivered without a sensitivity table — the implied price is extremely sensitive to both inputs. This grids WACC ±2% and terminal growth ±1% around the base case.

In [ ]:
def sensitivity_table(fcf_df: pd.DataFrame, mkt: dict, wacc_center: float, growth_center: float,
                       wacc_span: float = 0.02, growth_span: float = 0.01, steps: int = 5) -> pd.DataFrame:
    """Grid the implied share price across a range of WACC and terminal-growth assumptions."""
    wacc_range = np.linspace(wacc_center - wacc_span, wacc_center + wacc_span, steps)
    growth_range = np.linspace(growth_center - growth_span, growth_center + growth_span, steps)
    table = pd.DataFrame(
        index=[f"{w:.2%}" for w in wacc_range],
        columns=[f"{g:.2%}" for g in growth_range],
        dtype=float,
    )
    for w in wacc_range:
        for g in growth_range:
            table.loc[f"{w:.2%}", f"{g:.2%}"] = (
                np.nan if w <= g else dcf_valuation(fcf_df, w, g, mkt)["implied_price"]
            )
    table.index.name, table.columns.name = "WACC", "Terminal Growth"
    return table


sensitivity = sensitivity_table(fcf_projection, market, wacc_inputs["wacc"], TERMINAL_GROWTH)
display(sensitivity.style.background_gradient(cmap="RdYlGn", axis=None).format("${:,.0f}"))

## Section 8 — Visualizations

### 8.1 Historical Revenue & Operating Margin

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
years_lbl = historicals.index.year.astype(str)
ax1.bar(years_lbl, historicals["revenue"] / 1e9, color="#2563eb", label="Revenue ($B)")
ax1.set_ylabel("Revenue ($B)", color="#2563eb")
ax1.tick_params(axis="y", labelcolor="#2563eb")
ax1.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}B"))

ax2 = ax1.twinx()
ax2.plot(years_lbl, historicals["operating_margin"] * 100, color="#dc2626", marker="o", linewidth=2, label="Operating Margin (%)")
ax2.set_ylabel("Operating Margin (%)", color="#dc2626")
ax2.tick_params(axis="y", labelcolor="#dc2626")

plt.title(f"{TICKER} — Historical Revenue & Operating Margin", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### 8.2 Projected Free Cash Flow

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))
labels = [f"Y{y}" for y in fcf_projection["year"]]
ax1.bar(labels, fcf_projection["revenue"] / 1e9, color="#93c5fd", label="Projected Revenue ($B)")
ax1.set_ylabel("Revenue ($B)")

ax2 = ax1.twinx()
ax2.plot(labels, fcf_projection["fcf"] / 1e9, color="#16a34a", marker="o", linewidth=2, label="Unlevered FCF ($B)")
ax2.set_ylabel("Unlevered FCF ($B)", color="#16a34a")
ax2.tick_params(axis="y", labelcolor="#16a34a")

plt.title(f"{TICKER} — {PROJECTION_YEARS}-Year FCF Projection", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### 8.3 Interactive Sensitivity Heatmap

In [ ]:
fig = px.imshow(
    sensitivity.values,
    x=sensitivity.columns, y=sensitivity.index,
    labels=dict(x="Terminal Growth", y="WACC", color="Implied Price ($)"),
    color_continuous_scale="RdYlGn",
    text_auto=".0f",
    aspect="auto",
)
fig.update_layout(
    title=f"{TICKER} — Implied Share Price Sensitivity (WACC × Terminal Growth)",
    width=750, height=500,
)
fig.show()

### 8.4 DCF Implied Value vs. Current Market Price

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(
    ["DCF Implied Price", "Current Market Price"],
    [valuation["implied_price"], market["current_price"]],
    color=["#16a34a" if valuation["implied_price"] >= market["current_price"] else "#dc2626", "#6b7280"],
)
for bar in bars:
    width = bar.get_width()
    ax.text(width, bar.get_y() + bar.get_height() / 2, f" ${width:,.2f}", va="center", fontweight="bold")
ax.set_xlabel("Price per Share ($)")
ax.set_title(f"{TICKER} — DCF Implied Value vs. Market ({upside:+.1%})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Section 9 — Final Summary Report

In [ ]:
print("=" * 60)
print(f"  DCF VALUATION SUMMARY — {company_facts.get('entityName', TICKER)} ({TICKER})")
print("=" * 60)
print(f"  Risk-free rate (10Y UST):     {wacc_inputs['risk_free_rate']:.2%}")
print(f"  Beta:                         {market['beta']:.2f}")
print(f"  Cost of equity (CAPM):        {wacc_inputs['cost_of_equity']:.2%}")
print(f"  After-tax cost of debt:       {wacc_inputs['after_tax_cost_of_debt']:.2%}")
print(f"  WACC:                         {wacc_inputs['wacc']:.2%}")
print(f"  Terminal growth rate:         {TERMINAL_GROWTH:.2%}")
print("-" * 60)
print(f"  Enterprise value:             ${valuation['enterprise_value']:,.0f}")
print(f"  Equity value:                 ${valuation['equity_value']:,.0f}")
print(f"  Implied share price:          ${valuation['implied_price']:,.2f}")
print(f"  Current market price:         ${market['current_price']:,.2f}")
print(f"  Implied upside/(downside):    {upside:+.1%}")
print("=" * 60)

## Performance Metrics

There's no "accuracy" metric for a single-point valuation, but the model reports the metrics a real analyst would sanity-check before trusting it:
- **Implied vs. market price spread** (printed above) — how far the model's opinion is from the market's.
- **Sensitivity range** (Section 7) — how much the valuation swings across a plausible WACC/growth band; a very wide range signals a fragile, assumption-driven output rather than a robust intrinsic value.
- **Historical margin/growth stability** (Section 2) — noisy trailing-3-year averages make the whole forecast less trustworthy, which is worth calling out explicitly when presenting this.

## Final Deliverables

- A reusable, ticker-agnostic DCF pipeline (swap `TICKER` and re-run — no other code changes needed).
- A 6-year historical financial summary pulled live from SEC filings.
- A full WACC build-up (CAPM cost of equity + cost of debt).
- A 5-year unlevered FCF projection.
- An implied share price with an Enterprise Value → Equity Value bridge.
- A WACC × terminal-growth sensitivity table/heatmap.
- Four presentation-ready charts and a printed summary report.

## Resume Description

> *Built an automated DCF valuation engine in Python that pulls live financial statements from the SEC EDGAR XBRL API and market data from Yahoo Finance/FRED, computes CAPM-based WACC, projects unlevered free cash flow, and outputs an intrinsic share price with a WACC/terminal-growth sensitivity analysis — eliminating manual data entry from a traditional Excel DCF workflow.*

## Potential Upgrades

- Replace the flat historical-average margin assumption with a proper 3-statement model (working capital schedule, debt schedule, share count roll-forward).
- Add an exit-multiple terminal value method alongside Gordon growth and compare the two.
- Pull analyst consensus estimates (e.g. Financial Modeling Prep) to sanity-check the revenue/margin assumptions.
- Add a Monte Carlo layer that samples WACC, terminal growth, and margin assumptions from distributions instead of a fixed grid, and report a valuation range with a confidence interval.
- Extend to relative valuation (comps: EV/EBITDA, P/E) and blend with the DCF output for a football-field chart.
- Cache `company_facts` responses locally (e.g. to a `.json` or SQLite file) to avoid re-hitting SEC EDGAR on every re-run.

## Modeling Caveats & Limitations

- **This is a teaching/portfolio model, not investment advice.** Real DCFs (and this one) are extremely sensitive to growth, margin, and discount-rate assumptions — small changes swing the output a lot, as the sensitivity table above shows.
- XBRL tag coverage varies by company and filing era; `first_available_series()` mitigates this but doesn't eliminate it — always spot-check `historicals` against the company's actual 10-K before trusting the output.
- The net-working-capital assumption (10% of the change in revenue) is a simplification, not a modeled working-capital schedule.
- `yfinance` is an unofficial, delayed data source with no SLA — fine for a learning project, not for anything time-sensitive or production-grade.
- A DCF frequently undervalues high-growth/high-multiple compounders relative to the market price if margin expansion, buybacks, or multiple expansion aren't explicitly modeled — a large gap to the market price is informative, not necessarily a sign the model is broken.